In [ ]:
!pip install ultralytics --quiet

In [ ]:
!pip install roboflow --quiet

In [ ]:

import os
import cv2
import torch
from ultralytics import YOLO

In [ ]:
import os
from roboflow import Roboflow

 # Substitua <API-KEY-ROBOFLOW> pela sua chave de API do Roboflow
rf = Roboflow(api_key="<API-KEY-ROBOFLOW>") 

project = rf.workspace("yssp").project("disease-lettuce-h69zb")
version = project.version(4)

# 3. Download usando o formato "yolov8" (compatível com YOLOv11)
dataset = version.download("yolov8")

yaml_path = os.path.join(dataset.location, "data.yaml")
print(f"\n✅ Dataset baixado com sucesso!")
print(f"O arquivo de configuração está pronto para ser usado em: {yaml_path}")

# Verificando a estrutura de pastas para confirmar
print("\nEstrutura de pastas:")
!ls -R {dataset.location}

In [ ]:
#UTILIZADA SOMENTE SE O DATASET JÁ TIVER SIDO BAIXADO ANTERIORMENTE
import os
dataset_folder = "disease-lettuce-4" 
if os.path.exists(dataset_folder):
    print("✅ Dataset já baixado! Usando arquivos locais.")
    class Dataset:
        location = os.path.abspath(dataset_folder)
    dataset = Dataset()  
yaml_path = os.path.join(dataset.location, "data.yaml")

In [ ]:
import os

splits = ['train', 'valid', 'test']
contagem = {}

print("=" * 70)
print("📊 VERIFICAÇÃO DO DATASET - QUANTIDADE DE IMAGENS")
print("=" * 70)

for split in splits:
    images_path = os.path.join(dataset.location, split, 'images')
    labels_path = os.path.join(dataset.location, split, 'labels')

    if os.path.exists(images_path):
        # Contar imagens
        images = [f for f in os.listdir(images_path)
                 if f.endswith(('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'))]

        # Contar labels
        labels = []
        if os.path.exists(labels_path):
            labels = [f for f in os.listdir(labels_path) if f.endswith('.txt')]

        contagem[split] = {
            'images': len(images),
            'labels': len(labels)
        }

        print(f"\n📁 {split.upper()}:")
        print(f"   📸 Imagens: {len(images)}")
        print(f"   🏷️  Labels:  {len(labels)}")

        # Verificar se há imagens sem labels
        if len(images) != len(labels):
            print(f"   ⚠️  ATENÇÃO: {abs(len(images) - len(labels))} arquivo(s) desbalanceado(s)")
    else:
        print(f"\n📁 {split.upper()}: ❌ Pasta não encontrada")
        contagem[split] = {'images': 0, 'labels': 0}

# Totais
total_images = sum([contagem[s]['images'] for s in splits])
total_labels = sum([contagem[s]['labels'] for s in splits])

print("\n" + "=" * 70)
print(f"📦 TOTAL GERAL:")
print(f"   📸 Total de Imagens: {total_images}")
print(f"   🏷️  Total de Labels:  {total_labels}")
print("=" * 70)

# Análise da distribuição atual
print(f"\n📊 DISTRIBUIÇÃO ATUAL:")
for split in splits:
    if contagem[split]['images'] > 0:
        perc = (contagem[split]['images'] / total_images * 100)
        print(f"   {split.upper()}: {perc:.1f}%")

print("\n" + "=" * 70)

In [ ]:
# ============================================
# REBALANCEAR DATASET: 80% TREINO / 20% TESTE
# ============================================

import os
import shutil
import yaml
import random

random.seed(42)

print("=" * 70)
print("⚖️  REBALANCEAMENTO DO DATASET")
print("=" * 70)
print("📊 Nova distribuição: 80% TREINO / 20% TESTE")
print("=" * 70)

all_data = []

for split in ['train', 'valid', 'test']:
    images_path = os.path.join(dataset.location, split, 'images')
    labels_path = os.path.join(dataset.location, split, 'labels')

    if os.path.exists(images_path):
        for img_file in os.listdir(images_path):
            if img_file.endswith(('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')):
                img_path = os.path.join(images_path, img_file)
                label_file = os.path.splitext(img_file)[0] + '.txt'
                label_path = os.path.join(labels_path, label_file)

                # Só adiciona se tiver o label correspondente
                if os.path.exists(label_path):
                    all_data.append({
                        'image': img_path,
                        'label': label_path,
                        'filename': img_file
                    })

print(f"\n✅ Total de imagens coletadas: {len(all_data)}")

random.shuffle(all_data)

split_idx = int(len(all_data) * 0.8)
train_data = all_data[:split_idx]
test_data = all_data[split_idx:]

print(f"\n📦 NOVA DISTRIBUIÇÃO:")
print(f"   🟢 TREINO: {len(train_data)} imagens ({len(train_data)/len(all_data)*100:.1f}%)")
print(f"   🔵 TESTE:  {len(test_data)} imagens ({len(test_data)/len(all_data)*100:.1f}%)")

new_dataset_path = dataset.location + '_rebalanced'

if os.path.exists(new_dataset_path):
    print(f"\n🗑️  Removendo dataset rebalanceado anterior...")
    shutil.rmtree(new_dataset_path)

# Criar pastas (train e test, mas test será usado como val também)
for split in ['train', 'test']:
    os.makedirs(os.path.join(new_dataset_path, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(new_dataset_path, split, 'labels'), exist_ok=True)

# 5. Copiar arquivos para nova estrutura
print(f"\n📁 Copiando arquivos para: {new_dataset_path}")
print("   Aguarde...")

for item in train_data:
    dest_img = os.path.join(new_dataset_path, 'train', 'images', item['filename'])
    shutil.copy2(item['image'], dest_img)

    label_filename = os.path.splitext(item['filename'])[0] + '.txt'
    dest_label = os.path.join(new_dataset_path, 'train', 'labels', label_filename)
    shutil.copy2(item['label'], dest_label)

for item in test_data:
    dest_img = os.path.join(new_dataset_path, 'test', 'images', item['filename'])
    shutil.copy2(item['image'], dest_img)

    label_filename = os.path.splitext(item['filename'])[0] + '.txt'
    dest_label = os.path.join(new_dataset_path, 'test', 'labels', label_filename)
    shutil.copy2(item['label'], dest_label)

print("   ✅ Arquivos copiados com sucesso!")

original_yaml = os.path.join(dataset.location, 'data.yaml')
new_yaml = os.path.join(new_dataset_path, 'data.yaml')

if os.path.exists(original_yaml):
    with open(original_yaml, 'r') as f:
        data_config = yaml.safe_load(f)

    data_config['path'] = new_dataset_path
    data_config['train'] = 'train/images'
    data_config['val'] = 'test/images'  # 'val' para o Yolo
    data_config['test'] = 'test/images'

    with open(new_yaml, 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

    print(f"\n📄 Arquivo data.yaml criado em: {new_yaml}")

print("\n" + "=" * 70)
print("✅ REBALANCEAMENTO CONCLUÍDO COM SUCESSO!")
print("=" * 70)
print(f"\n📂 Novo dataset: {new_dataset_path}")
print(f"📄 Novo YAML: {new_yaml}")

print("\n🚀 PRÓXIMOS PASSOS:")
print("   Use este código para treinar:")
print(f"   yaml_path = '{new_yaml}'")
print("   results = model.train(data=yaml_path, epochs=40, imgsz=640, batch=16, workers=4, device=0)")

yaml_path = new_yaml
print(f"\n✅ Variável 'yaml_path' atualizada automaticamente!")

In [ ]:
#UTILIZAR SOMENTE SE O DATASET REBALANCEADO JÁ TIVER SIDO ATUALIZADO ANTERIORMENTE
import os
dataset_folder = "disease-lettuce-4_rebalanced" 
if os.path.exists(dataset_folder):
    print("✅ Dataset já baixado! Usando arquivos locais.")
    class Dataset:
        location = os.path.abspath(dataset_folder)
    dataset = Dataset()  
yaml_path = os.path.join(dataset.location, "data.yaml")

✅ Dataset já baixado! Usando arquivos locais.


In [ ]:
from ultralytics import YOLO, RTDETR
import os

modelos_para_baixar = [
    ("YOLO11l", "yolo11l.pt"),
    ("RT-DETR-l", "rtdetr-l.pt")
]

for nome, arquivo in modelos_para_baixar:
    if os.path.exists(arquivo):
        print(f"✅ {nome}: Arquivo '{arquivo}' já existe no disco.")
    else:
        print(f"⏳ {nome}: Baixando '{arquivo}'...")
        try:

            if "rtdetr" in arquivo:
                RTDETR(arquivo)
            else:
                YOLO(arquivo)
            print(f"   -> Download concluído com sucesso!")
        except Exception as e:
            print(f"❌ ERRO ao baixar {nome}: {e}")

print("\n📦 Tudo pronto!")

In [4]:
import torch
print(f"Versão do PyTorch: {torch.__version__}")
print(f"CUDA Disponível? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Placa detectada: {torch.cuda.get_device_name(0)}")
    print("✅ Tudo pronto para o treino!")
else:
    print("❌ Ainda está na CPU. Tente reiniciar o Kernel.")

Versão do PyTorch: 2.6.0+cu124
CUDA Disponível? True
Placa detectada: NVIDIA GeForce RTX 4050 Laptop GPU
✅ Tudo pronto para o treino!


In [3]:
import gc
import torch
from ultralytics import YOLO, RTDETR

TRAIN_ARGS = {
    'data':  yaml_path,     
    'epochs': 50,
    'patience': 25,
    'imgsz': 640,
    'device': 0,
    'workers': 2,          
    'project': 'treinamento_lettuce_results',
    'seed': 42,
    'save': True,
    'plots': True,
    'exist_ok': True,
    'amp': True,            
    'batch': 4,           
}

modelos = [
    {'nome': 'YOLO11l',   'arquivo': 'yolo11l.pt'}, 
    {'nome': 'RT-DETR-l', 'arquivo': 'rtdetr-l.pt'},
]

print(f"🚀 Iniciando bateria de testes com Batch 4...")

for i, config in enumerate(modelos):
    print(f"\n{'='*60}")
    print(f"({i+1}/{len(modelos)}) TREINANDO: {config['nome']}")
    print(f"{'='*60}")

    try:
        if 'rtdetr' in config['arquivo']:
            model = RTDETR(config['arquivo'])
        else:
            model = YOLO(config['arquivo'])
        
        results = model.train(**TRAIN_ARGS, name=config['nome'])
        print(f"✅ Sucesso: {config['nome']} finalizado!")

    except Exception as e:
        print(f"ERRO CRÍTICO no modelo {config['nome']}: {e}")

    print(" Limpeza memória da GPU...")
    if 'model' in locals(): del model
    if 'results' in locals(): del results
    gc.collect()
    torch.cuda.empty_cache()
    print("Memória limpa.\n")

print("Treinamento finalizado.")

🚀 Iniciando bateria de testes com Batch 4...

(1/2) TREINANDO: YOLO11l
Ultralytics 8.3.235  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\projeto_ia\disease-lettuce-4_rebalanced\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=YOLO11l, nbs=64, nms=Fal

In [ ]:
import gc
import torch
from ultralytics import RTDETR

torch.backends.cudnn.benchmark = True

# Configuração ESPECÍFICA para o RT-DETR 
TRAIN_ARGS = {
    'data':  yaml_path,     
    'epochs': 50,
    'patience': 25,
    'imgsz': 512,
    'device': 0,
    'workers': 2,          
    'project': 'treinamento_lettuce_results',
    'seed': 42,
    'save': True,
    'plots': True,
    'exist_ok': True,
    'amp': True,            
    'batch': 8,
    'cache': True,          
}

print(f"🚀 Iniciando RT-DETR-l...")

try:
    model = RTDETR('rtdetr-l.pt')
    
    print(f"\n{'='*60}") 
    print(f"TREINANDO: RT-DETR-l")
    print(f"{'='*60}")
    
    results = model.train(**TRAIN_ARGS, name='RT-DETR-l')
    print(f"Sucesso: RT-DETR-l finalizado!")

except Exception as e:
    print(f"ERRO CRÍTICO PERSISTENTE: {e}")

print("🧹 Limpando memória...")
if 'model' in locals(): del model
if 'results' in locals(): del results
gc.collect()
torch.cuda.empty_cache()
print("Pronto.")

🚀 Iniciando RT-DETR-l...



TREINANDO: RT-DETR-l
New https://pypi.org/project/ultralytics/8.3.236 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.235  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\projeto_ia\disease-lettuce-4_rebalanced\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1

In [7]:
from ultralytics import YOLO
import os

model_path = r'treinamento_lettuce_results\YOLO11l\weights\best.pt'

model = YOLO(model_path)

print("INICIANDO BATERIA DE TESTES FINAL")
print("="*60)

print("📊 Calculando métricas oficiais (mAP, Precision, Recall)...")
metrics = model.val(
    data=yaml_path,
    split='test',       
    imgsz=640,
    batch=4,            
    conf=0.25,         
    iou=0.6,         
    plots=True,      
    save_json=False
)

print(f"\nPara Análise:")
print(f"   - mAP50 (Precisão Média): {metrics.box.map50:.4f}")
print(f"   - mAP50-95 (Rigoroso):    {metrics.box.map:.4f}")
print(f"   - Precision:              {metrics.box.mp:.4f}")
print(f"   - Recall:                 {metrics.box.mr:.4f}")


INICIANDO BATERIA DE TESTES FINAL
📊 Calculando métricas oficiais (mAP, Precision, Recall)...
Ultralytics 8.3.235  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)


YOLO11l summary (fused): 190 layers, 25,284,709 parameters, 0 gradients, 86.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 258.7280.7 MB/s, size: 82.7 KB)
val: Scanning C:\projeto_ia\disease-lettuce-4_rebalanced\test\labels.cache... 1371 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1371/1371 721.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 343/343 7.4it/s 46.5s<0.1s
                   all       1371       3016      0.642      0.609      0.633      0.496
             Bacterial        181        247      0.892      0.972      0.952      0.802
          Downy_mildew        368        475      0.694      0.756      0.754      0.648
 Lettuce - Anthracnose         79        100      0.429       0.36      0.356      0.284
        Powdery_mildew         84        108      0.566      0.398      0.471      0.277
       Septoria_Blight        354        486      0.751      0.794      0.823      0.702
     

In [2]:
from ultralytics import YOLO
model_path = r'treinamento_lettuce_results\YOLO11l\weights\best.pt'

model = YOLO(model_path)

In [ ]:
print("Gerando imagens visuais das predições...")

test_images_path = os.path.join(dataset.location, "test", "images")

results = model.predict(
    source=test_images_path,
    conf=0.35,         
    iou=0.5,
    save=True,        
    imgsz=640,
    device=0
)

print(f"\nCONCLUÍDO!")
print(f" Gráficos e Métricas em: {metrics.save_dir}")
print(f" Fotos com detecções em: {results[0].save_dir}")

In [ ]:
from ultralytics import RTDETR, YOLO
import cv2

video_path = r'videos\PROJETO HORTA\horta1.MP4'
output_name = 'resultado_1280p'   

#  Carregar Modelo
# try:
#     model = RTDETR(model_path)
# except:
#     model = YOLO(model_path)

print(f"Processando vídeo 4K na RTX 4050...")

results = model.predict(
    source=video_path,
    save=True,
    imgsz=1280,   
    conf=0.25,   
    iou=0.45,   
    device=0,        
    stream=True   
)

for result in results:
    pass 

print(f"\n✅ Vídeo salvo na pasta runs/detect/predict...")

Processando vídeo 4K na RTX 4050...

video 1/1 (frame 1/2169) c:\projeto_ia\videos\PROJETO HORTA\horta1.MP4: 736x1280 1 Bacterial, 2 Septoria_Blights, 276.3ms
video 1/1 (frame 2/2169) c:\projeto_ia\videos\PROJETO HORTA\horta1.MP4: 736x1280 1 Bacterial, 2 Septoria_Blights, 95.6ms
video 1/1 (frame 3/2169) c:\projeto_ia\videos\PROJETO HORTA\horta1.MP4: 736x1280 1 Bacterial, 2 Septoria_Blights, 90.3ms
video 1/1 (frame 4/2169) c:\projeto_ia\videos\PROJETO HORTA\horta1.MP4: 736x1280 1 Septoria_Blight, 88.6ms
video 1/1 (frame 5/2169) c:\projeto_ia\videos\PROJETO HORTA\horta1.MP4: 736x1280 1 Bacterial, 2 Septoria_Blights, 91.4ms
video 1/1 (frame 6/2169) c:\projeto_ia\videos\PROJETO HORTA\horta1.MP4: 736x1280 2 Septoria_Blights, 91.2ms
video 1/1 (frame 7/2169) c:\projeto_ia\videos\PROJETO HORTA\horta1.MP4: 736x1280 2 Septoria_Blights, 38.8ms
video 1/1 (frame 8/2169) c:\projeto_ia\videos\PROJETO HORTA\horta1.MP4: 736x1280 1 Bacterial, 2 Septoria_Blights, 38.8ms
video 1/1 (frame 9/2169) c:\projet

In [ ]:
from ultralytics import RTDETR, YOLO
import cv2

video_path = r'videos\PROJETO HORTA\horta2.MP4'
output_name = 'resultado_1280p'   

#  Carregar Modelo
# try:
#     model = RTDETR(model_path)
# except:
#     model = YOLO(model_path)

print(f"Processando vídeo 4K na RTX 4050...")

results = model.predict(
    source=video_path,
    save=True,
    imgsz=1280,   
    conf=0.25,   
    iou=0.45,   
    device=0,        
    stream=True   
)

for result in results:
    pass 

print(f"\n✅ Vídeo salvo na pasta runs/detect/predict...")

Processando vídeo 4K na RTX 4050...

video 1/1 (frame 1/4148) c:\projeto_ia\videos\PROJETO HORTA\horta2.MP4: 736x1280 1 healthy, 377.9ms
video 1/1 (frame 2/4148) c:\projeto_ia\videos\PROJETO HORTA\horta2.MP4: 736x1280 (no detections), 76.8ms
video 1/1 (frame 3/4148) c:\projeto_ia\videos\PROJETO HORTA\horta2.MP4: 736x1280 (no detections), 79.2ms
video 1/1 (frame 4/4148) c:\projeto_ia\videos\PROJETO HORTA\horta2.MP4: 736x1280 (no detections), 76.1ms
video 1/1 (frame 5/4148) c:\projeto_ia\videos\PROJETO HORTA\horta2.MP4: 736x1280 1 healthy, 111.4ms
video 1/1 (frame 6/4148) c:\projeto_ia\videos\PROJETO HORTA\horta2.MP4: 736x1280 1 healthy, 68.6ms
video 1/1 (frame 7/4148) c:\projeto_ia\videos\PROJETO HORTA\horta2.MP4: 736x1280 1 healthy, 80.0ms
video 1/1 (frame 8/4148) c:\projeto_ia\videos\PROJETO HORTA\horta2.MP4: 736x1280 1 healthy, 78.9ms
video 1/1 (frame 9/4148) c:\projeto_ia\videos\PROJETO HORTA\horta2.MP4: 736x1280 1 healthy, 79.1ms
video 1/1 (frame 10/4148) c:\projeto_ia\videos\PROJE

In [ ]:
import glob
from IPython.display import Image, display

results_folder = 'runs/detect/predict'

image_files = glob.glob(f'{results_folder}/*.jpg')

print(f"Exibindo as primeiras {min(5, len(image_files))} imagens de um total de {len(image_files)} resultados:")
print("-" * 30)

for image_path in image_files[:5]:
    print(f"Arquivo: {image_path}")
    display(Image(filename=image_path, width=600))
    print("\n")

In [ ]:
import os
import glob
import yaml
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

if 'yaml_path' in locals() and os.path.exists(yaml_path):
    with open(yaml_path, 'r') as file:
        data_info = yaml.safe_load(file)
        class_names = data_info['names']
    print(f"Nomes das classes carregados: {class_names}")
else:
    print("ERRO: Execute a célula de download do dataset primeiro para definir 'yaml_path'.")
    class_names = []

results_folder = 'runs/detect/predict'
predicted_images = glob.glob(f'{results_folder}/*.jpg')

if not predicted_images:
    print(f"Nenhuma imagem encontrada na pasta de resultados: {results_folder}")
else:
    image_index_to_inspect = 7

    if image_index_to_inspect >= len(predicted_images):
        print(f"Índice fora do alcance. Por favor, escolha um número entre 0 e {len(predicted_images)-1}.")
    elif 'dataset' not in locals():
        print("ERRO: A variável 'dataset' não foi encontrada. Execute a célula de download primeiro.")
    else:
        
        predicted_image_path = predicted_images[image_index_to_inspect]
        predicted_image = Image.open(predicted_image_path)
        base_filename = os.path.basename(predicted_image_path)

        
        original_image_path = os.path.join(dataset.location, "valid/images", base_filename)
        label_path = os.path.join(dataset.location, "valid/labels", os.path.splitext(base_filename)[0] + ".txt")

        
        original_image = cv2.imread(original_image_path)
        h, w, _ = original_image.shape
        ground_truth_image = original_image.copy()

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    class_id, x_center, y_center, width, height = map(float, line.split())
                    class_id = int(class_id)
                    x1 = int((x_center - width / 2) * w)
                    y1 = int((y_center - height / 2) * h)
                    x2 = int((x_center + width / 2) * w)
                    y2 = int((y_center + height / 2) * h)

                    text_y = y1 - 10
                    
                    if text_y < 20:
                        text_y = y1 + 25 

                    
                    cv2.rectangle(ground_truth_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(ground_truth_image, class_names[class_id], (x1, text_y),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 3)
        else:
            print(f"Arquivo de rótulo não encontrado para {base_filename}")

        ground_truth_image = cv2.cvtColor(ground_truth_image, cv2.COLOR_BGR2RGB)

       
        fig, ax = plt.subplots(1, 2, figsize=(20, 10))
        ax[0].imshow(ground_truth_image)
        ax[0].set_title(f'Gabarito (Ground Truth)\nArquivo: {base_filename}', fontsize=14)
        ax[0].axis('off')

        ax[1].imshow(predicted_image)
        ax[1].set_title('Previsão do Modelo', fontsize=14)
        ax[1].axis('off')

        plt.show()